In [ ]:
import gc
import glob
from tqdm import tqdm
from utils.utils import *
from utils.metric import *
from models.modernTCN import ModernTCN
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath
from torch.utils.data import DataLoader, TensorDataset
from modules.diffusion import TimeSeriesGaussianDiffusion, generate_linear_schedule, generate_scaled_linear_schedule


file_manager = quantDiffusionPath()
gc.collect()

# Preprocessing

In [ ]:
df = pd.read_csv(f"./data/indices.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df.set_index('Date', inplace=True)
df = df.apply(pd.to_numeric).astype(float)

seq_len = 256
log_returns = np.log1p(df.pct_change()).dropna().values
log_returns_windowed = torch.from_numpy(rolling_window(log_returns, seq_len)).float()
log_returns_scaled, scalers = scaling(log_returns)
log_returns_scaled_windowed = torch.from_numpy(rolling_window(log_returns_scaled, seq_len)).float()
input_tensor = log_returns_scaled_windowed.permute(0, 2, 1).clone().detach()
log_returns_inversed = inverse_scaling(input_tensor, scalers)
log_returns_inversed_windowed = torch.from_numpy(log_returns_inversed.transpose(0, 2, 1)).float()

print(log_returns_windowed.shape)
print(log_returns_scaled_windowed.shape)
print(log_returns_inversed_windowed.shape)

init_price = torch.from_numpy(np.array(df)[:-(seq_len), :]).float().unsqueeze(1)

In [ ]:
if len(log_returns_scaled_windowed.shape) == 2:
    # (N, 6) -> (N-127, 128, 6)
    log_returns_scaled_windowed = rolling_window(log_returns_scaled_windowed, seq_len)
else:
    log_returns_scaled_windowed = log_returns_scaled_windowed
    print('Already windowed')

# 2. 데이터 개수 맞추기 (Alignment)
# 윈도우 데이터와 시작 가격(init_price)의 개수를 동기화합니다.
min_size = min(log_returns_scaled_windowed.shape[0], init_price.shape[0])

log_returns_final = log_returns_scaled_windowed[:min_size]
init_price_final = init_price[:min_size]

# 3. 인덱스 셔플링
perm_idx = np.random.permutation(min_size)

# 4. 데이터 분할
train_size = int(0.7 * min_size)

# [B, T, C] 형태 유지
training_data = torch.tensor(log_returns_final[perm_idx[:train_size]]).float()
train_init_price = torch.tensor(init_price_final[perm_idx[:train_size]]).float()

test_data = torch.tensor(log_returns_final[perm_idx[train_size:]]).float()
test_init_price = torch.tensor(init_price_final[perm_idx[train_size:]]).float()

print(f"✅ 최종 데이터 형상: {training_data.shape}") # 예상: [Batch, 128, 6]

# 1. 데이터를 파이토치 텐서로 변환 (이미 텐서라면 생략 가능)
# 넘파이 배열인 경우 torch.from_numpy()를 사용하는 것이 효율적입니다.
if not isinstance(training_data, torch.Tensor):
    train_tensor = torch.from_numpy(training_data).float()
    test_tensor = torch.from_numpy(test_data).float()
else:
    train_tensor = training_data.float()
    test_tensor = test_data.float()

# 2. TensorDataset 생성
# 데이터가 하나만 들어가는 경우에도 텐서 형태로 전달해야 합니다.
training_set = TensorDataset(train_tensor)
test_set = TensorDataset(test_tensor)

# 3. DataLoader 설정
train_dl = DataLoader(training_set, batch_size=256, shuffle=True)

print("✅ DataLoader 준비 완료!")

In [ ]:
configs = {
    "enc_in": 6,
    # 1. 채널 수를 32의 배수로 정렬: [96, 192, 96]
    # 중간 층(Hidden layer)을 192로 확장하여 특징 추출 능력을 극대화했습니다.
    "dims": [96, 128, 96], 
    # "dims": [128, 512, 128], 
    
    # 2. 커널 사이즈 (수용 영역 유지)
    "large_size": [15, 7, 5], 
    "small_size": [5, 3, 3],
    
    # 3. 시간 조건화 차원 (2의 배수)
    "time_emb_dim": 256, 
    
    "kernel_size": 51, 
    "small_kernel_merged": False,
    "revin": False,
    "affine": True,
    "decomposition": False,
    "use_time_emb": True
}

# configs = {
#     "enc_in": 6,
#     # 1. 채널 수 조정: 총 파라미터를 약 39.7만 개로 설계
#     # [32, 64, 128, 96, 64, 32] 구조로 병목(Bottleneck)을 형성하여 효율을 높였습니다.
#     "dims": [32, 64, 128, 96, 64, 32], 
    
#     # 2. 커널 및 수용 영역: 6개 레이어로 155일의 Receptive Field 확보
#     "large_size": [15, 7, 5, 3, 3, 3], 
#     "small_size": [5, 3, 3, 3, 3, 3],
    
#     # 3. 기타 고정 설정
#     "time_emb_dim": 256, 
#     "small_kernel_merged": False,
#     "revin": False,
#     "affine": False,
#     "decomposition": False,
#     "use_time_emb": True
# }

ts_model = ModernTCN(configs)

# ts_model = ModernTCN2(configs)
# 모델 인스턴스를 전달하여 실행


first_batch = next(iter(train_dl))
data = first_batch[0] 
t_batch = torch.tensor([99], device='cpu').repeat(256)
output = ts_model(data, t_batch)

# ---------- Time Series DDPM 학습 ----------
# 위에서 정의한 train_dl, ts_model(ModernTCN), test_data 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = training_data.shape[1]   # 128
num_channels = training_data.shape[2]  # 6
num_timesteps = 4000

os.makedirs(file_manager.get_weight_path(), exist_ok=True)
# 학습 설정
num_epochs = 250
log_every = 10
train_iter = 0

# 스케줄 및 diffusion 생성
# betas = generate_linear_schedule(num_timesteps, 2.5e-5, 0.005)
# betas = generate_linear_schedule(num_timesteps, 1e-4, 0.02)
betas = generate_scaled_linear_schedule(num_timesteps, 1e-4, 0.02)
diffusion = TimeSeriesGaussianDiffusion(
    ts_model,
    seq_len=seq_len,
    num_channels=num_channels,
    betas=betas,
    loss_type="l2",
    ema_decay=0.9999,
    ema_start=2000,
    ema_update_rate=1,
    sampling_type='student_t',
).to(device)

optimizer = torch.optim.Adam(diffusion.parameters(), lr=2e-4)

count_parameters(ts_model)
print(diffusion.sampling_type)

# Model Training

In [ ]:
weight_dir = file_manager.get_weight_path()
start_epoch = 0
ckpts = glob.glob(os.path.join(weight_dir, "quantDiffusion_[0-9]*.pt"))

if ckpts:
    def _epoch_from_path(p):
        stem = os.path.basename(p).replace("quantDiffusion_", "").replace(".pt", "")
        try:
            return int(stem)
        except ValueError:
            return -1
    epochs_found = [e for e in (_epoch_from_path(p) for p in ckpts) if e >= 0]
    if epochs_found:
        resume_epoch = max(epochs_found)
        start_epoch = resume_epoch
        latest_ckpt = os.path.join(weight_dir, f"quantDiffusion_{resume_epoch}.pt")
        diffusion.load_state_dict(torch.load(latest_ckpt, map_location=device))
        print(f"Resumed from {latest_ckpt}, continuing from epoch {start_epoch}")

pbar = tqdm(range(start_epoch, num_epochs))

for epoch in pbar:
    diffusion.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for batch in train_dl:
        x = batch[0].to(device)
        b = x.shape[0]
        t = torch.randint(0, diffusion.num_timesteps, (b,), device=x.device, dtype=torch.long)

        loss = diffusion(x)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        diffusion.update_ema()
        
        current_loss = loss.item()
        epoch_loss += current_loss
        n_batches += 1
        train_iter += 1

    avg_loss = epoch_loss / n_batches
    pbar.set_description(f"Epoch {epoch+1}")
    pbar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if (epoch % log_every == 0) and (epoch != 0):
        pbar.set_postfix(iter=train_iter, loss=f"{current_loss:.4f}")
        torch.save(diffusion.state_dict(), f'{file_manager.get_weight_path()}/quantDiffusion_{epoch}.pt')

# Model Validation

In [ ]:
result_path = f"{file_manager.get_result_path(diffusion.sampling_type)}"
os.makedirs(result_path, exist_ok=True)  # exist_ok=True로 중복 에러 방지

validate = True
if validate:
    min_dist = float('inf')  
    pbar = tqdm(range(50, num_epochs, log_every))

    for epoch in pbar:

        # Load generator for the current epoch
        diffusion.load_state_dict(torch.load(f'{file_manager.get_weight_path()}/quantDiffusion_{epoch}.pt'))
        diffusion.to(device)
        diffusion.eval()
        
        fake = diffusion.sample(batch_size=100, device=device, use_ema=True)
        # print(fake.transpose(1, 2).shape)
        # print(training_data.transpose(1, 2).shape)
        fake_data = inverse_scaling(fake.transpose(1, 2), scalers)        
        real_data = inverse_scaling(training_data.transpose(1, 2), scalers)

        # Compute EMD
        emd = compute_avg_emd(real_data, fake_data, 100)  
            
            
        # Update best epoch if current EMD is lower
        if emd < min_dist:
            min_dist = emd
            best_epoch = epoch
            print(f"min_distance: {min_dist:.5f}, best_epoch: {best_epoch}") 

        else:
            print(f"current_distance: {emd:.5f}, current_epoch: {epoch}")

    diffusion.load_state_dict(torch.load(f'{file_manager.get_weight_path()}/quantDiffusion_{best_epoch}.pt', map_location=device))
    torch.save(diffusion.state_dict(), f'{result_path}/quantDiffusion_best({best_epoch}).pt')

    # 1. 가중치가 저장된 폴더 경로 및 검색 패턴 설정
    weight_path = file_manager.get_weight_path()
    search_pattern = os.path.join(weight_path, 'quantDiffusion_*.pt')

    # 2. 패턴과 일치하는 모든 파일 리스트업
    files_to_delete = glob.glob(search_pattern)

    # 3. 파일 삭제 진행
    if files_to_delete:
        print(f"총 {len(files_to_delete)}개의 가중치 파일을 삭제합니다...")
        for file_path in files_to_delete:
            try:
                os.remove(file_path)
            except Exception as e:
                print(f"❌ 삭제 실패 ({os.path.basename(file_path)}): {e}")    
        print("✅ 모든 가중치 파일 정리가 완료되었습니다.")
    else:
        print("🧹 삭제할 가중치 파일이 존재하지 않습니다.")

# Save synthetic data

In [ ]:
search_pattern = f'{result_path}/quantDiffusion_best*.pt'
best_files = glob.glob(search_pattern)

if best_files:
    target_file = sorted(best_files)[-1]
    diffusion.load_state_dict(torch.load(target_file))
    print(f"✅ 가중치 로드 완료: {os.path.basename(target_file)}")
else:
    print("❌ 해당 경로에 'quantDiffusion_best*.pt' 패턴의 파일이 없습니다.")

diffusion.to(device)
diffusion.eval()

real_data = inverse_scaling(test_data.transpose(1, 2), scalers)
# --- [2. 안전한 이상치 필터링 경계값 계산] ---
# 실제 데이터의 각 피처별 최소/최대값을 구합니다.
min_vals = real_data.min(axis=(0, 2), keepdims=True)
max_vals = real_data.max(axis=(0, 2), keepdims=True)

# 단순 곱셈이 아닌, 실제 데이터의 '진폭(Range)'을 기준으로 50%의 여유 공간을 둡니다.
range_margin = (max_vals - min_vals) * 0.5 
lower_bound = min_vals - range_margin
upper_bound = max_vals + range_margin


# --- [3. 데이터 생성 루프] ---
for i in range(5):
    print(f'\n--- Sample generation {i+1}/5 ---')
    
    # 데이터 생성 (두 배치 생성 후 병합)
    samples1 = diffusion.sample(batch_size=250, device=device, use_ema=True)
    samples2 = diffusion.sample(batch_size=250, device=device, use_ema=True)
    samples = torch.cat([samples1, samples2], dim=0)
    print('✅ Generation Completed!')

    fake_data = inverse_scaling(samples.transpose(1, 2), scalers)        

    # Compute EMD
    emd = compute_avg_emd(real_data, fake_data, 100)
    print(f"📊 min_distance (EMD): {emd:.5f}")

    # 마스크 계산 (안전한 경계값 적용)
    mask = np.all((fake_data >= lower_bound) & (fake_data <= upper_bound), axis=(1, 2))
    filtered_fake_data = fake_data[mask]
    
    # 저장
    np.save(os.path.join(result_path, f'{i}.npy'), filtered_fake_data)

# Evaluation

In [ ]:
result_path = f"{file_manager.get_result_path(diffusion.sampling_type)}"
real_data = inverse_scaling(test_data.transpose(1, 2), scalers)
total_result_df=pd.DataFrame()
for seed in range(5):
    fake_data = np.load(f"{result_path}/{seed}.npy")
    fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
    real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]
    x = fake_data.transpose(0, 2, 1)
    flat_x = x.reshape(-1, x.shape[-1])

    y = real_data.transpose(0, 2, 1)
    flat_y = y.reshape(-1, y.shape[-1])

    """ Visualize the distribution of the real and fake data """

    # Plot the distribution of the real and fake data
    windows = [1, 5, 20, 100]

    result_df=pd.DataFrame()
    CCF_df = calculate_ccf(real_data, fake_data, 0)
    result_df = pd.concat([result_df, CCF_df], axis=0)
    total_CCF_df=pd.DataFrame(columns=['Average'])
    for i in range(1,11):
        total_CCF_df = pd.concat([total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1)

    result_df = pd.concat([result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=['Average'])], axis=0)

    similarity_df = calculate_distribution_scores(real_data, fake_data, windows)
    result_df = pd.concat([result_df, similarity_df], axis=0)
    skew_kurt_df = calculate_skew_kurtosis(real_data, fake_data, windows)
    result_df = pd.concat([result_df, skew_kurt_df], axis=0)
    _, acf_df = calculate_acf_score(real_data, fake_data, 40)
    result_df = pd.concat([result_df, acf_df], axis=0)
    _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
    result_df = pd.concat([result_df, leverage_df], axis=0)

    total_result_df = pd.concat([total_result_df, result_df], axis=1)

statistics_df = pd.DataFrame(index=total_result_df.index, columns=['MEAN', 'STD'])
statistics_df.loc[statistics_df.index, 'MEAN']=total_result_df.mean(axis=1).values
statistics_df.loc[statistics_df.index, 'STD']=total_result_df.std(axis=1).values
statistics_df.to_csv(file_manager.get_result_file(f"{diffusion.sampling_type}_gen_quality"))

In [ ]:
import yaml
import ml_collections
from utils.eval_portfolio.summary import *
from utils.eval_portfolio.strategies import log_return_to_price

eval_size = real_data.shape[0]

fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)
real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)

test_init_price = train_init_price
test_init_price[:] = 100

# 1. 원본 데이터를 보존하기 위해 clone()을 사용하고, 100으로 초기화합니다.
# (test_init_price = train_init_price 라고 하면 원본 값도 100으로 바뀝니다)
test_init_price = train_init_price.clone()
test_init_price[:] = 100

# 2. 평가할 실제 데이터의 양을 정확히 계산합니다.
# fake_data_torch가 eval_size보다 작을 경우를 대비합니다.
actual_eval_data = fake_data_torch[:eval_size]
n_samples = actual_eval_data.shape[0]

# 3. 데이터 개수(n_samples)에 맞춰서 초기 가격도 똑같이 잘라줍니다.
fake_prices = log_return_to_price(actual_eval_data, test_init_price[:n_samples])
real_prices = log_return_to_price(real_data_torch[:n_samples], test_init_price[:n_samples])


config_dir = f'{LOCAL_BASE_DIRECTORY}/utils/eval_portfolio/config.yaml'
with open(config_dir) as file:
    eval_config = ml_collections.ConfigDict(yaml.safe_load(file))


all_positive = (fake_prices > 0).all()
if not all_positive:
    raise ValueError("Sanity Check Failed: Some fake prices are not positive.")

res_dict = {
    "var_abs_mean": 0.,
    "var_rel_mean": 0.,
    "es_abs_mean": 0.,
    "es_rel_mean": 0.,
}


# 1. 설정 및 변수 준비
alphas = eval_config.alphas  # [0.05, 0.025, 0.01]
strat_names = ['equal_weight', 'mean_reversion', 'trend_following', 'vol_trading']
# 표시용 약어 매핑
strat_map = {
    'equal_weight': 'EW',
    'mean_reversion': 'MR',
    'trend_following': 'TF',
    'vol_trading': 'VT'
}

# 결과를 저장할 중첩 딕셔너리 {Metric(alpha): {Strategy: Value}}
final_results = {}

with torch.no_grad():
    for alpha in alphas:
        # Config의 alpha 설정 동기화
        eval_config.Evaluation.TestMetrics.var.alpha = alpha
        eval_config.Evaluation.TestMetrics.es.alpha = alpha
        
        # 각 Metric별로 전략들의 수치를 담을 임시 딕셔너리
        var_row = {}
        es_row = {}

        for strat_name in strat_names:
            # 전략별 평가 수행
            subres_dict, _, _ = full_evaluation(
                fake_prices, 
                real_prices, 
                eval_config, 
                strat_name=strat_name
            )
            
            # 약어 이름으로 개별 전략 결과 저장
            short_name = strat_map[strat_name]
            var_row[short_name] = round(subres_dict.get('var_rel_mean', 0.), 4)
            es_row[short_name] = round(subres_dict.get('es_rel_mean', 0.), 4)
            
        # 4개 전략의 평균(Average) 계산 및 추가
        var_row['Average'] = round(np.mean(list(var_row.values())), 4)
        es_row['Average'] = round(np.mean(list(es_row.values())), 4)
        
        # 최종 결과 사전에 인덱스 명칭으로 저장
        final_results[f"RE_VaR({alpha})"] = var_row
        final_results[f"RE_ES({alpha})"] = es_row
        

# 2. DataFrame 포맷팅
# Metric 순서 정의 (VaR 그룹 후 ES 그룹)
metrics = ['RE_VaR', 'RE_ES']
ordered_keys = [f"{m}({a})" for m in metrics for a in alphas]

# 딕셔너리로부터 DataFrame 생성 및 컬럼 순서 정렬
df_final = pd.DataFrame.from_dict(final_results, orient='index')
cols_order = [strat_map[s] for s in strat_names] + ['Average']
df_final = df_final.reindex(index=ordered_keys, columns=cols_order)
df_final.index.name = 'Metric'
df_final.T.to_csv(file_manager.get_result_file(f'{diffusion.sampling_type}_portfolio'))

In [ ]:
# 상관행렬 계산을 위해 데이터 전치 (shape: (num_samples, num_time_steps, num_features))
real_mean_corr = correlation_mean(np.transpose(real_data, (0, 2, 1)))
fake_mean_corr = correlation_mean(np.transpose(fake_data, (0, 2, 1)))

feature_names = ['DJI', 'IXIC', 'JPM', 'HSI', 'Gold', 'WTI']

plot_correlation_heatmap(real_mean_corr, "Real Data", feature_names, diffusion.sampling_type)
plot_correlation_heatmap(fake_mean_corr, "Fake Data", feature_names, diffusion.sampling_type)
plot_cumulative_return_dist(real_list, fake_list, windows=windows, diffusion.sampling_type)
plot_acf_comparison(real_list, fake_list, 6, lags=100, diffusion.sampling_type)